# Answer Key

This answer key contains a copy of each exercise text as well as the solutions, but you will not be able to run solution cells here. Copy each solution from this answer key into the main notebook for this learning module and run the code there. 

<div class="alert alert-info"> 
    
## Exercise 1: Read, check CRS, and plot shapefile data

In this exercise you'll practice using GeoPandas to read and display information from a shapefile.

Use GeoPandas to read the file ```data/admin_boundaries/state_boundaries/FL_GA_SC_NC_cb_2023_us_state_500k.shp``` into a GeoDataFrame called ```states``` and display the data rows.

</div>

In [ ]:
# add your code here
states = gpd.read_file('data/admin_boundaries/state_boundaries/FL_GA_SC_NC_cb_2023_us_state_500k.shp')
states

<div class="alert alert-info"> 

What is the CRS of the ```states``` GeoDataFrame and is it a geographic CRS or a projected CRS? 
</div>

In [ ]:
# add your code here
states.crs

Type answer here: The CRS is EPSG:4269 and it is a geographic CRS.

<div class="alert alert-info"> 

Make a simple plot of the state shapes in the ```states``` GeoDataFrame.
</div>

In [ ]:
# add your code here
states.plot()

<div class="alert alert-info"> 

Make a simple plot of the state shapes in the ```states``` GeoDataFrame in CRS EPSG:5070.
</div>

In [ ]:
# add your code here
states.to_crs('EPSG:5070').plot()

<div class="alert alert-info"> 

## Exercise 2: GeoSeries attributes

In this exercise you will use the ```states``` GeoDataFrame that you created in Exercise 1 and practice using GeoSeries attributes to get information about the shapes in the GeoDataFrame. 

First, get the Shapely geometry object for South Carolina from the GeoDataFrame into a variable called ```sc_shape```, display the Shapely object in your code cell output. Also, programmatically show that ```sc_shape``` is, in fact, a Shapely geometry object.
</div>

In [ ]:
# add your code here
sc_shape = states.loc[1,'geometry']
sc_shape

In [ ]:
# add your code here
type(sc_shape)

<div class="alert alert-info"> 

Get the minx, miny, maxx, maxy bounds only for the state of Florida and save the resulting GeoDataFrame to a new variable called ```fl_bounds```. Display the resulting row of data.
</div>

In [ ]:
# add your code here

fl_bounds = states.loc[states.STUSPS=='FL'].bounds
fl_bounds

<div class="alert alert-info"> 

Get only the miny value for the state of Florida as a single float data value, not a GeoSeries or GeoDataFrame object.
</div>

In [ ]:
# add your code here
fl_bounds.loc[2,'miny']

In [ ]:
# alternatively
fl_bounds.reset_index(drop=True).loc[0,'miny']

<div class="alert alert-info"> 

Reproject the ```states``` GeoDataFrame to EPSG:5070 and save the result in a new variable called ```states_5070```.
</div>

In [ ]:
# add your code here

states_5070 = states.to_crs('EPSG:5070')

states_5070.crs

<div class="alert alert-info"> 

Calculate the area of each state and save the result to a new column in ```states_5070``` called ```area```. Display all the rows in ```states_5070```. What are the units of the area column?
</div>

In [ ]:
# add your code here
states_5070['area'] = states_5070.area
states_5070

Type answer here: Units for areas in EPSG:5070 are square meters.

<div class="alert alert-info"> 

## Exercise 3: Creating buffers

Create a 100m buffer around each superfund site and save the results to ```gdf_sites``` in a new column called BUFF_100M.

</div>

In [ ]:
# add your code here
gdf_sites['BUFF_100M']=gdf_sites.buffer(100)
gdf_sites.head()

<div class="alert alert-info"> 

Visualize the Chemfax site and site buffer. Make the site buffer your base plot and then overlay the site polygon in orange.
</div>

In [ ]:
# add your code here
base = gdf_sites.loc[[3]].BUFF_100M.plot()
gdf_sites.loc[[3]].plot(ax=base,color='orange')

<div class="alert alert-info"> 

## Exercise 4: GeoPandas .within()

Use the example above of applying a custom function to a GeoDataFrame and test whether each superfund site CENTROID is within any of the 1000 meter river buffers (buffer_1000m column in ```gdf_rivers```). Save your result to a new column in ```gdf_sites``` called CEN_IN_RIVBUF. Show the CEN_IN_RIVBUF column results.



In [ ]:
# add your code here

def centroid_within_riverbuffer(row, rivers):
    return row.CENTROID.within(rivers.buffer_1000m).any()     

gdf_sites['CEN_IN_RIVBUF'] = gdf_sites.apply(centroid_within_riverbuffer, axis=1, args=(gdf_rivers,))
gdf_sites.CEN_IN_RIVBUF

<div class="alert alert-info"> 

## Exercise 5: Find sites that intersect a river buffer

Test whether each site polygon (geometry column in ```gdf_sites```) lies at least partially within any of the river buffers (buffer_1000m column in ```gdf_rivers```) and return your results to a new column in ```gdf_sites``` called ```INTERSECTS_RIVBUF```. You will need to write a new custom function that uses ```.intersects()``` and apply it to each row of the ```gdf_sites``` GeoDataFrame. 

</div>

In [ ]:
# add you code here

def site_intersects_riverbuffer(row, rivers):
    return row.geometry.intersects(rivers.buffer_1000m).any()

gdf_sites['INTERSECTS_RIVBUF'] = gdf_sites.apply(site_intersects_riverbuffer, axis=1, args=(gdf_rivers,))
gdf_sites

<div class="alert alert-info"> 

Use ```.loc[]``` to show only the data rows with sites that are at least partially in a 1000 meter river buffer.
</div>

In [ ]:
# add your code here
gdf_sites.loc[gdf_sites.INTERSECTS_RIVBUF==True]

<div class="alert alert-info"> 

Programatically find the integer number of sites that are at least partially in a 1000 meter river buffer and save the result to a new variable called ```nsites```.
</div>

In [ ]:
# add your code here
nsites = gdf_sites.INTERSECTS_RIVBUF.sum()
nsites

In [ ]:
# alternatively
nsites = gdf_sites.loc[gdf_sites.INTERSECTS_RIVBUF==True].shape[0]
nsites

<div class="alert alert-info"> 

# XIV. Exercise: Putting It All Together

Use GeoPandas to read, manipulate, analyze, and visualize data from shapefiles of world countries and cities. 

## A) Read a shapefile into a GeoDataFrame and visualize

### read shapefile

The file ```data/admin_boundaries/county_boundaries/tl_2020_us_county_subset.shp``` contains the county boundaries in five states (MS, LA, AR, AL, TN) from the US Census Bureau. Load it into in a GeoDataFrame called ```counties```. Print a preview of the rows in ```counties```.

In [ ]:
# add your code here
counties = gpd.read_file('data/admin_boundaries/county_boundaries/tl_2020_us_county_subset.shp')
counties.head()

<div class="alert alert-info"> 

How many data rows are in ```counties```?

In [ ]:
# add your code here
counties.shape

Type your answer: 383

<div class="alert alert-info"> 

Are the columns in ```counties``` numeric or non-numeric data types?

In [ ]:
# add your code here
counties.dtypes

Type your answer: all columns are non-numeric

<div class="alert alert-info"> 

What is the CRS of ```counties``` and what are the units? Is it a geographic or projected CRS?

In [ ]:
# add your code here
counties.crs

Type your answer: EPSG 4269 with units in degrees, which is a geographic CRS

<div class="alert alert-info"> 

### visualize polygons

Make a simple plot of ```counties```.

In [ ]:
# add your code here
counties.plot()

<div class="alert alert-info"> 

## B) Create a GeoDataFrame from a .txt file and visualize

### read data from a text file

The file ```data/population/us_cb_2020_counties/centers_of_population_by_county.txt``` contains the geographic center of population for each county in the US. The data in the file is comma delimited. Read it into a Pandas DataFrame called ```pop_df```. Print a preview of data rows.

In [ ]:
# add your code here
pop_df = pd.read_csv('data/population/us_cb_2020_counties/centers_of_population_by_county.txt')
pop_df.head()

<div class="alert alert-info"> 

How many data rows are in ```pop_df```?

In [ ]:
# add your code here
pop_df.shape

Type your answer: 3221 data rows

<div class="alert alert-info"> 

Are the columns in ```pop_df``` numeric or non-numeric data types?

In [ ]:
# add your code here
pop_df.dtypes

Type your answer: all numeric except COUNAME and STNAME which are non-numeric

<div class="alert alert-info"> 

### generate point shapes from latitudes and longitudes

Use GeoPandas to generate points from the LATITUDE and LONGITUDE columns in ```pop_df```. Use the same CRS as the ```counties``` GeoDataFrame. Save your points in a variable called ```geometry```.

In [ ]:
# generate vector data points
geometry = gpd.points_from_xy(pop_df['LONGITUDE'], pop_df['LATITUDE'], crs=counties.crs)
geometry

<div class="alert alert-info"> 

### convert DataFrame to GeoDataFrame

Use your variables ```pop_df``` and ```geometry``` to create a GeoDataFrame called ```pop_gdf```. Print the CRS, column data types, and a preview of data rows in ```pop_gdf```.

In [ ]:
# add your code here
pop_gdf = gpd.GeoDataFrame(pop_df, geometry=geometry)

print(pop_gdf.crs)
print(pop_gdf.dtypes)
pop_gdf.head()

<div class="alert alert-info"> 

### visualize points 

Make a simple plot of the population center points in ```pop_gdf```. This plot doesn't need to look nice, the goal is simply to see the spatial extent of the points.

In [ ]:
# add your code here
pop_gdf.plot()

<div class="alert alert-info"> 

## C) Reproject GeoDataFrames

We'll be making buffers later in the exercise, therefore we will reproject to a projected CRS with cartesian coordinates. Reproject ```counties``` and ```pop_gdf``` to EPSG 5070.

In [ ]:
# add your code here
counties = counties.to_crs('EPSG:5070')
pop_gdf = pop_gdf.to_crs('EPSG:5070')

print(counties.crs)
pop_gdf.crs

<div class="alert alert-info"> 

What units does EPSG 5070 have?

Type your answer: meters

<div class="alert alert-info"> 

## D) Perform a spatial join on two GeoDataFrames and visualize

### spatial join

The goal is to join the population and geometry information from ```pop_gdf``` to ```counties```.

Start by copying the geometry in ```pop_gdf``` to new column in ```pop_gdf``` called POP_CENTER. 

In [ ]:
# add your code here
pop_gdf['POP_CENTER'] = pop_gdf['geometry']
pop_gdf.columns

<div class="alert alert-info"> 

Now use GeoPandas to join only the STNAME, POPULATION, POP_CENTER, and geometry columns from ```pop_gdf``` to ```counties``` where the geometries intersect. Save the result to a new variable called ```county_pop```.

**Hints:** This is a ```geopandas.sjoin()``` procedure with ```how='inner'```. Use the double plain bracket syntax to select specific columns from ```pop_gdf``` in the join. There should be 383 rows in your result.

In [ ]:
# add your code here
county_pop = gpd.sjoin(counties,pop_gdf[['STNAME','POPULATION','POP_CENTER','geometry']],how='inner')
county_pop

<div class="alert alert-info"> 

### visualize population by county

Make a map where counties are colored by population (cloropleth map). Include a horizontal colorbar labeled "Population in 2020").

In [ ]:
# add your code here
county_pop.plot(column='POPULATION', legend=True,
                legend_kwds={"label": "Population in 2020", "orientation": "horizontal"})

<div class="alert alert-info"> 

### visualize county boundaries and population centers

Using your ```county_pop``` GeoDataFrame, make a plot with the county boundaries in black and the population centers in blue. Do not fill the counties (only use black borders) and reduce the thickness of the borders with the parameter ```lw=0.5```. For the population centers use the parameter ```marker='.'```.

In [ ]:
# add your code here
base = county_pop.plot(facecolor='none', edgecolor='black',lw=0.5)
county_pop.POP_CENTER.plot(ax=base, marker='.')

<div class="alert alert-info"> 

## E) Use dissolve to find state population

Use the columns STNAME, POPULATION, and geometry in ```county_pop``` to dissolve the county shapes into state shapes and sum the population by state. Save the result to a new variable called ```state_pop```.

**Hint:** Summing a column during a dissolve is something we haven't seen yet. Use the example on the GeoPandas website where ```nepal_pop``` is dissolved by zone and population is summed with the ```aggfunc``` parameter: https://geopandas.org/en/stable/docs/user_guide/aggregation_with_dissolve.html

In [ ]:
# add your code here
state_pop = county_pop[['STNAME','POPULATION','geometry']].dissolve(by='STNAME',as_index=False,sort=False,aggfunc='sum')
state_pop

<div class="alert alert-info"> 

Programmatically show the state names for the states with the highest and lowest population.

**Hint:** Use ```.idxmax()``` and ```.idxmin()```.

In [ ]:
# add your code here
print(f'The most populous state is {state_pop.loc[state_pop.POPULATION.idxmax(),'STNAME']}.')
print(f'The least populous state is {state_pop.loc[state_pop.POPULATION.idxmin(),'STNAME']}.')

<div class="alert alert-info"> 

## F) Create buffers

Create a 15km buffer around each county population center in Mississippi and save the buffers to a new column in ```county_pop``` called BUFF_15KM.

In [ ]:
# add your code here
county_pop['BUFF_15KM'] = county_pop.loc[county_pop.STNAME=='Mississippi','POP_CENTER'].buffer(15000)
county_pop

<div class="alert alert-info"> 

## G) Find all buffers that intersect other states and visualize

### identify buffers with multi-state intersections

This task is a bit more complex than the others. The goal is to create a new column called MULTISTATE in ```county_pop``` that contains a value of True or False based on whether each 15km buffer intersects with any state that is not Mississippi. Write a custom function and use ```.apply()``` to accomplish this.

**Hints:**
- Use ```.apply()``` only on the rows of ```county_pop``` where there is a shape in the BUFF_15KM column. This means your apply should look something like ```county_pop.loc[conditional expression].apply(parameters)```. You can select the appropriate rows inside ```.loc[]``` with a combination of bitwise not and the ```.isna()``` function.
- In your custom function, test whether the buffer intersects each state shape in ```state_pop``` and sum that result. If a buffer intersects more than one state, your sum will be greater than 1 and your custom function should return True. If a buffer only intersects Mississippi, your sum will be 1 and your custom function should return False. You'll want to use an if-else statement to make your custom function return the appropriate result. 

In [ ]:
# add your code here
def buff_intersects_states(row,states):
    if row.BUFF_15KM.intersects(states.geometry).sum() > 1:
        return True
    else:
        return False
        
county_pop['MULTISTATE'] = county_pop.loc[~county_pop.BUFF_15KM.isna()].apply(buff_intersects_states,axis=1,args=(state_pop,))
county_pop

<div class="alert alert-info"> 

### visualize

Make a figure showing the county boundaries in black, the population centers whose 15km buffers are fully within Mississippi in blue, and the population centers whose 15km buffers extend beyond Mississippi in orange. Use ```lw=0.5``` for the county boundaries and ```marker='.'``` for all points.

In [ ]:
# add your code here
base = county_pop.plot(facecolor='none', edgecolor='black',lw=0.5)
county_pop.loc[county_pop.MULTISTATE==False,'POP_CENTER'].plot(ax=base, marker='.')
county_pop.loc[county_pop.MULTISTATE==True,'POP_CENTER'].plot(ax=base, marker='.')